<a href="https://colab.research.google.com/github/eya-sheikhrouhou/Ipssi-Projects/blob/API/Mini_Projet1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Clé API
La clé API est stockée dans les Secrets de Colab (icône 🔑) et non dans le code,
pour qu'elle ne soit pas visible si le notebook est partagé.

In [ ]:
from google.colab import userdata

# On récupère la clé depuis les Secrets de Colab
API_KEY = userdata.get('TMDB_API_KEY')

# Adresse de base de l'API TMDB
BASE_URL = "https://api.themoviedb.org/3"

# On vérifie que la clé est chargée, sans l'afficher en entier
print("Clé chargée :", API_KEY[:4] + "****")

Clé chargée : 2634****


## 2. Première requête à l'API
On interroge l'endpoint /movie/{id} pour le film 550 afin d'observer la structure de la réponse.

In [ ]:
import requests

# 1. On construit l'adresse complète
url = f"{BASE_URL}/movie/550"

# 2. On envoie la demande, avec la clé et la langue en paramètres
response = requests.get(url, params={"api_key": API_KEY, "language": "fr-FR"})

# 3. On regarde si ça a marché
print("Code de réponse :", response.status_code)

# 4. On transforme la réponse en dictionnaire Python et on l'affiche
data = response.json()
data

Code de réponse : 200


{'adult': False,
 'backdrop_path': '/c6OLXfKAk5BKeR6broC8pYiCquX.jpg',
 'belongs_to_collection': None,
 'budget': 63000000,
 'genres': [{'id': 18, 'name': 'Drame'}, {'id': 53, 'name': 'Thriller'}],
 'homepage': '',
 'id': 550,
 'imdb_id': 'tt0137523',
 'origin_country': ['US'],
 'original_language': 'en',
 'original_title': 'Fight Club',
 'overview': 'Le narrateur, sans identité précise, vit seul, travaille seul, dort seul, mange seul ses plateaux‐repas pour une personne comme beaucoup d’autres personnes seules qui connaissent la misère humaine, morale et sexuelle. C’est pourquoi il va devenir membre du Fight club, un lieu clandestin où il va pouvoir retrouver sa virilité, l’échange et la communication. Ce club est dirigé par Tyler Durden, une sorte d’anarchiste entre gourou et philosophe qui prêche l’amour de son prochain.',
 'popularity': 42.3856,
 'poster_path': '/t1i10ptOivG4hV7erkX3tmKpiqm.jpg',
 'production_companies': [{'id': 711,
   'logo_path': '/tEiIH5QesdheJmDAqQwvtN60727.pn

## 3. Fonction get_movie_details
Fonction qui prend un id de film et renvoie : titre, date de sortie, genres, popularité et note moyenne.

In [ ]:
def get_movie_details(movie_id):
    """Renvoie les infos principales d'un film à partir de son id TMDB."""

    # 1. On construit l'adresse avec l'id reçu en paramètre
    url = f"{BASE_URL}/movie/{movie_id}"

    # 2. On envoie la demande
    response = requests.get(url, params={"api_key": API_KEY, "language": "fr-FR"})

    # 3. On transforme la réponse en dictionnaire
    data = response.json()

    # 4. On extrait seulement les noms des genres
    genres = [genre["name"] for genre in data["genres"]]

    # 5. On renvoie seulement les infos qui nous intéressent
    return {
        "titre": data["title"],
        "date_sortie": data["release_date"],
        "genres": genres,
        "popularite": data["popularity"],
        "note_moyenne": data["vote_average"],
    }


# Test avec le film 550
get_movie_details(550)

{'titre': 'Fight Club',
 'date_sortie': '1999-10-15',
 'genres': ['Drame', 'Thriller'],
 'popularite': 42.3856,
 'note_moyenne': 8.4}

In [ ]:
get_movie_details(999999999)

KeyError: 'genres'

Test avec un film inexistant : la fonction plante (KeyError)

## 4. Q1 – Fonction de requête robuste
Fonction générale qui gère les erreurs courantes :
- erreurs temporaires (429, 500, 502, 503, 504, timeout, réseau) : pause puis nouvelle tentative
  avec un délai qui double à chaque essai (backoff exponentiel) ;
- erreurs définitives (400, 401, 403, 404) : arrêt immédiat avec un message clair.

In [ ]:
import time
import requests

# Codes pour lesquels ça vaut la peine de réessayer
CODES_TEMPORAIRES = {429, 500, 502, 503, 504}

# Codes définitifs : inutile de réessayer
MESSAGES_ERREURS = {
    400: "Requête mal formée (400) : vérifie les paramètres envoyés",
    401: "Non autorisé (401) : clé API absente ou invalide",
    403: "Accès interdit (403) : tu n'as pas les droits sur cette ressource",
    404: "Introuvable (404) : cet élément n'existe pas",
}

def requete_robuste(url, params=None, max_tentatives=5, pause_initiale=1, timeout=10):
    """Envoie une requête GET en gérant les erreurs courantes.
    Renvoie le JSON de la réponse, ou None en cas d'échec."""

    for tentative in range(1, max_tentatives + 1):
        # Temps d'attente si on doit réessayer : 1s, 2s, 4s, 8s...
        attente = pause_initiale * 2 ** (tentative - 1)

        # 1. Envoyer la requête en attrapant les problèmes réseau
        try:
            response = requests.get(url, params=params, timeout=timeout)
        except requests.exceptions.Timeout:
            print(f"⏳ Tentative {tentative} : pas de réponse (timeout), nouvel essai dans {attente}s")
            time.sleep(attente)
            continue
        except requests.exceptions.ConnectionError:
            print(f"⏳ Tentative {tentative} : problème de connexion, nouvel essai dans {attente}s")
            time.sleep(attente)
            continue

        code = response.status_code

        # 2. Succès
        if code == 200:
            return response.json()

        # 3. Erreur temporaire : on attend et on réessaie
        if code in CODES_TEMPORAIRES:
            # Pour le 429, le serveur indique parfois combien de temps attendre
            if code == 429 and "Retry-After" in response.headers:
                attente = int(response.headers["Retry-After"])
            print(f"⏳ Tentative {tentative} : erreur {code}, nouvel essai dans {attente}s")
            time.sleep(attente)
            continue

        # 4. Erreur définitive : on arrête
        print("❌", MESSAGES_ERREURS.get(code, f"Erreur inattendue ({code})"))
        return None

    print(f"❌ Échec après {max_tentatives} tentatives")
    return None

In [ ]:
def get_movie_details_v3(movie_id, language="fr-FR"):
    """Infos d'un film, en s'appuyant sur requete_robuste."""

    # Vérifier l'identifiant avant d'appeler l'API
    try:
        movie_id = int(movie_id)
        if movie_id <= 0:
            raise ValueError
    except (TypeError, ValueError):
        print(f"❌ Identifiant non valide : {movie_id!r}")
        return None

    data = requete_robuste(
        f"{BASE_URL}/movie/{movie_id}",
        params={"api_key": API_KEY, "language": language},
    )
    if data is None:
        return None

    return {
        "id": data.get("id"),
        "titre": data.get("title"),
        "date_sortie": data.get("release_date"),
        "genres": [g["name"] for g in data.get("genres", [])],
        "popularite": data.get("popularity"),
        "note_moyenne": data.get("vote_average"),
    }

In [ ]:
print("=== Cas normaux sur TMDB ===")
print(get_movie_details_v3(550))
print(get_movie_details_v3(999999999))   # 404
print(get_movie_details_v3("abc"))       # identifiant non valide

print("\n=== Clé invalide ===")
print(requete_robuste(f"{BASE_URL}/movie/550", params={"api_key": "fausse_cle"}))  # 401

print("\n=== Simulation des autres erreurs avec httpbin ===")
for code in [400, 403, 429, 503]:
    print(f"\n--- Test du code {code} ---")
    requete_robuste(f"https://httpbin.org/status/{code}", max_tentatives=3)

=== Cas normaux sur TMDB ===
{'id': 550, 'titre': 'Fight Club', 'date_sortie': '1999-10-15', 'genres': ['Drame', 'Thriller'], 'popularite': 42.3856, 'note_moyenne': 8.4}
❌ Introuvable (404) : cet élément n'existe pas
None
❌ Identifiant non valide : 'abc'
None

=== Clé invalide ===
❌ Non autorisé (401) : clé API absente ou invalide
None

=== Simulation des autres erreurs avec httpbin ===

--- Test du code 400 ---
❌ Requête mal formée (400) : vérifie les paramètres envoyés

--- Test du code 403 ---
❌ Accès interdit (403) : tu n'as pas les droits sur cette ressource

--- Test du code 429 ---
⏳ Tentative 1 : erreur 429, nouvel essai dans 1s
⏳ Tentative 2 : erreur 429, nouvel essai dans 2s
⏳ Tentative 3 : erreur 429, nouvel essai dans 4s
❌ Échec après 3 tentatives

--- Test du code 503 ---
⏳ Tentative 1 : erreur 503, nouvel essai dans 1s
⏳ Tentative 2 : erreur 503, nouvel essai dans 2s
⏳ Tentative 3 : erreur 503, nouvel essai dans 4s
❌ Échec après 3 tentatives


## 5. Q2 – Gérer un très grand nombre de requêtes (asyncio)
Problème : en séquentiel, chaque requête attend la précédente (1 million × 0,15 s ≈ 42 h).
Solution : asyncio permet de lancer de nombreuses requêtes en parallèle. Pendant qu'une requête
attend sa réponse, les autres avancent. Un Semaphore limite le nombre de requêtes simultanées
pour ne pas surcharger le serveur.
Les URLs api.example.com n'existant pas, on simule un serveur (délai aléatoire + erreurs aléatoires).

In [ ]:
import asyncio
import random

URLS = [
    "https://api.example.com/data1",
    "https://api.example.com/data2",
    "https://api.example.com/data3",
]

async def faux_serveur(url):
    """Simule une API : un temps de réponse variable et quelques erreurs aléatoires."""

    # Le serveur met entre 0,05 et 0,2 seconde à répondre
    await asyncio.sleep(random.uniform(0.05, 0.2))

    # On tire un nombre au hasard entre 0 et 1 pour décider de la réponse
    tirage = random.random()
    if tirage < 0.05:
        return 429      # 5 % : trop de requêtes
    if tirage < 0.08:
        return 503      # 3 % : serveur indisponible
    if tirage < 0.09:
        return 404      # 1 % : introuvable
    return 200          # le reste : succès

In [ ]:
async def requete_robuste_async(url, semaphore, max_tentatives=5, pause_initiale=0.1):
    """Version asynchrone de requete_robuste (même logique de retry que la Q1)."""

    async with semaphore:   # on attend qu'une "place" soit libre
        for tentative in range(1, max_tentatives + 1):
            code = await faux_serveur(url)

            if code == 200:
                return {"url": url, "statut": "succès", "tentatives": tentative}

            if code in CODES_TEMPORAIRES:   # 429, 503... (défini en Q1)
                await asyncio.sleep(pause_initiale * 2 ** (tentative - 1))
                continue

            return {"url": url, "statut": f"erreur {code}", "tentatives": tentative}

        return {"url": url, "statut": "échec", "tentatives": max_tentatives}

In [ ]:
import time

async def lancer_sequentiel(n):
    """Une requête à la fois (comme avant)."""
    semaphore = asyncio.Semaphore(1)     # 1 seule place = une par une
    resultats = []
    for i in range(n):
        resultats.append(await requete_robuste_async(URLS[i % 3], semaphore))
    return resultats

async def lancer_concurrent(n, max_simultanees=1000):
    """Beaucoup de requêtes en même temps."""
    semaphore = asyncio.Semaphore(max_simultanees)
    taches = [requete_robuste_async(URLS[i % 3], semaphore) for i in range(n)]
    return await asyncio.gather(*taches)


# Comparaison sur 100 requêtes
debut = time.perf_counter()
await lancer_sequentiel(100)
duree_seq = time.perf_counter() - debut
print(f"🐢 Séquentiel : 100 requêtes en {duree_seq:.2f} s")

debut = time.perf_counter()
await lancer_concurrent(100)
duree_conc = time.perf_counter() - debut
print(f"🚀 Concurrent : 100 requêtes en {duree_conc:.2f} s")

print(f"➡️ {duree_seq / duree_conc:.0f} fois plus rapide")

🐢 Séquentiel : 100 requêtes en 15.67 s
🚀 Concurrent : 100 requêtes en 0.39 s
➡️ 40 fois plus rapide


In [ ]:
from collections import Counter

N = 100_000   # 100 000 requêtes

debut = time.perf_counter()
resultats = await lancer_concurrent(N, max_simultanees=1000)
duree = time.perf_counter() - debut

print(f"✅ {N:,} requêtes en {duree:.1f} s, soit {N / duree:,.0f} requêtes par seconde")
print("\nRépartition par statut :", Counter(r["statut"] for r in resultats))
print("Répartition par route  :", Counter(r["url"] for r in resultats))
print("Requêtes ayant nécessité un retry :", sum(r["tentatives"] > 1 for r in resultats))

print(f"\n➡️ Estimation pour 1 million de requêtes : environ {1_000_000 / (N / duree) / 60:.1f} minutes")

✅ 100,000 requêtes en 17.5 s, soit 5,705 requêtes par seconde

Répartition par statut : Counter({'succès': 98911, 'erreur 404': 1088, 'échec': 1})
Répartition par route  : Counter({'https://api.example.com/data1': 33334, 'https://api.example.com/data2': 33333, 'https://api.example.com/data3': 33333})
Requêtes ayant nécessité un retry : 8036

➡️ Estimation pour 1 million de requêtes : environ 2.9 minutes


## 6. Q3 – Cache avec expiration (TTL = 1 heure)
Le cache garde en mémoire les réponses déjà obtenues. Si la même requête est refaite
dans l'heure, on renvoie la réponse stockée (cache hit) sans appeler l'API.
Au-delà d'1 heure, la réponse est considérée comme périmée et on rappelle l'API (cache miss).
Avantages : moins d'appels à l'API (moins de charge serveur, moins de risques de 429)
et des réponses beaucoup plus rapides.

In [ ]:
import time

class CacheExpirant:
    """Cache en mémoire où chaque réponse expire après `duree_vie` secondes."""

    def __init__(self, duree_vie=3600):
        self.duree_vie = duree_vie   # 3600 s = 1 heure
        self.donnees = {}            # { clé : (valeur, heure_de_stockage) }
        self.hits = 0                # nombre de fois où le cache a servi
        self.misses = 0              # nombre de fois où il a fallu appeler l'API

    def get(self, cle):
        """Renvoie la valeur si elle est en cache et pas périmée, sinon None."""
        if cle in self.donnees:
            valeur, heure_stockage = self.donnees[cle]
            age = time.time() - heure_stockage
            if age < self.duree_vie:
                self.hits += 1
                return valeur
            del self.donnees[cle]    # périmée : on la supprime
        self.misses += 1
        return None

    def set(self, cle, valeur):
        """Stocke une valeur avec l'heure actuelle."""
        self.donnees[cle] = (valeur, time.time())

    def stats(self):
        total = self.hits + self.misses
        taux = 100 * self.hits / total if total else 0
        return f"{self.hits} hits / {self.misses} misses (taux de réussite du cache : {taux:.1f} %)"

In [ ]:
# On crée UN cache, partagé par tous les appels
cache_tmdb = CacheExpirant(duree_vie=3600)

def requete_avec_cache(url, params=None, cache=cache_tmdb):
    """requete_robuste (Q1) + cache : on n'appelle l'API que si nécessaire."""

    # 1. La clé du cache = l'adresse + les paramètres (même requête = même clé)
    cle = url + str(sorted((params or {}).items()))

    # 2. On regarde d'abord dans le cache
    resultat = cache.get(cle)
    if resultat is not None:
        return resultat

    # 3. Pas dans le cache : on appelle l'API avec notre fonction robuste
    resultat = requete_robuste(url, params)

    # 4. On ne garde en cache que les succès (pas les erreurs)
    if resultat is not None:
        cache.set(cle, resultat)
    return resultat


def get_movie_details_v4(movie_id, language="fr-FR"):
    """Infos d'un film : robuste (Q1) + cache (Q3)."""
    try:
        movie_id = int(movie_id)
        if movie_id <= 0:
            raise ValueError
    except (TypeError, ValueError):
        print(f"❌ Identifiant non valide : {movie_id!r}")
        return None

    data = requete_avec_cache(
        f"{BASE_URL}/movie/{movie_id}",
        params={"api_key": API_KEY, "language": language},
    )
    if data is None:
        return None

    return {
        "id": data.get("id"),
        "titre": data.get("title"),
        "date_sortie": data.get("release_date"),
        "genres": [g["name"] for g in data.get("genres", [])],
        "popularite": data.get("popularity"),
        "note_moyenne": data.get("vote_average"),
    }

In [ ]:
print("=== 1er appel : le cache est vide, on interroge l'API ===")
debut = time.perf_counter()
print(get_movie_details_v4(550))
duree_1 = time.perf_counter() - debut
print(f"⏱️ {duree_1 * 1000:.1f} ms")

print("\n=== 2e appel identique : la réponse vient du cache ===")
debut = time.perf_counter()
print(get_movie_details_v4(550))
duree_2 = time.perf_counter() - debut
print(f"⏱️ {duree_2 * 1000:.3f} ms")

print(f"\n➡️ {duree_1 / duree_2:,.0f} fois plus rapide")

print("\n=== 5 films demandés 3 fois chacun ===")
for _ in range(3):
    for film in [550, 155, 680, 13, 27205]:
        get_movie_details_v4(film)
print(cache_tmdb.stats())

=== 1er appel : le cache est vide, on interroge l'API ===
{'id': 550, 'titre': 'Fight Club', 'date_sortie': '1999-10-15', 'genres': ['Drame', 'Thriller'], 'popularite': 42.3856, 'note_moyenne': 8.4}
⏱️ 66.6 ms

=== 2e appel identique : la réponse vient du cache ===
{'id': 550, 'titre': 'Fight Club', 'date_sortie': '1999-10-15', 'genres': ['Drame', 'Thriller'], 'popularite': 42.3856, 'note_moyenne': 8.4}
⏱️ 0.129 ms

➡️ 516 fois plus rapide

=== 5 films demandés 3 fois chacun ===
12 hits / 5 misses (taux de réussite du cache : 70.6 %)


In [ ]:
cache_test = CacheExpirant(duree_vie=3)   # 3 secondes au lieu de 3600

url_test = f"{BASE_URL}/movie/550"
params_test = {"api_key": API_KEY}

print("1. Premier appel →", "API" if cache_test.get("x") is None else "cache")
requete_avec_cache(url_test, params_test, cache=cache_test)

print("2. Appel immédiat :")
requete_avec_cache(url_test, params_test, cache=cache_test)
print("  ", cache_test.stats())

print("3. On attend 4 secondes (plus que la durée de vie)...")
time.sleep(4)

requete_avec_cache(url_test, params_test, cache=cache_test)
print("4. Après expiration :", cache_test.stats())
print("➡️ La réponse avait expiré : l'API a été rappelée (1 miss de plus)")

1. Premier appel → API
2. Appel immédiat :
   1 hits / 2 misses (taux de réussite du cache : 33.3 %)
3. On attend 4 secondes (plus que la durée de vie)...
4. Après expiration : 1 hits / 3 misses (taux de réussite du cache : 25.0 %)
➡️ La réponse avait expiré : l'API a été rappelée (1 miss de plus)


In [ ]:
cache_async = CacheExpirant(duree_vie=3600)
nb_appels_serveur = 0

async def requete_async_avec_cache(url, semaphore, cache):
    """Version asynchrone (Q2) + cache (Q3)."""
    global nb_appels_serveur

    resultat = cache.get(url)
    if resultat is not None:
        return resultat                     # ⚡ cache hit : pas d'appel serveur

    nb_appels_serveur += 1
    resultat = await requete_robuste_async(url, semaphore)
    if resultat["statut"] == "succès":
        cache.set(url, resultat)
    return resultat


def url_ressource(j):
    """Adresse de la ressource n°j, réparties sur les 3 routes."""
    return f"{URLS[j % 3]}?id={j}"


semaphore = asyncio.Semaphore(1000)

# Vague 1 : 100 ressources différentes, le cache est vide
debut = time.perf_counter()
await asyncio.gather(*[requete_async_avec_cache(url_ressource(j), semaphore, cache_async)
                       for j in range(100)])
print(f"Vague 1 : 100 requêtes en {time.perf_counter() - debut:.2f} s, "
      f"{nb_appels_serveur} appels au serveur")

# Vague 2 : 100 000 requêtes qui redemandent ces mêmes 100 ressources
nb_appels_serveur = 0
debut = time.perf_counter()
await asyncio.gather(*[requete_async_avec_cache(url_ressource(i % 100), semaphore, cache_async)
                       for i in range(100_000)])
print(f"Vague 2 : 100 000 requêtes en {time.perf_counter() - debut:.2f} s, "
      f"{nb_appels_serveur:,} appels au serveur")

print("\n", cache_async.stats())

Vague 1 : 100 requêtes en 0.87 s, 100 appels au serveur
Vague 2 : 100 000 requêtes en 3.56 s, 0 appels au serveur

 100000 hits / 100 misses (taux de réussite du cache : 99.9 %)


## 7. Export quotidien des identifiants de films
TMDB publie chaque jour un fichier .gz contenant les identifiants de tous ses films.
On télécharge celui de la date courante (ou de la veille s'il n'est pas encore publié),
puis on le décompresse pour afficher son contenu.

In [ ]:
from datetime import date, timedelta

def telecharger_export_films(nb_jours_max=3):
    """Télécharge l'export des films du jour (ou des jours précédents si absent)."""

    for decalage in range(nb_jours_max + 1):
        # 1. La date : aujourd'hui, puis hier, puis avant-hier...
        jour = date.today() - timedelta(days=decalage)

        # 2. Le nom du fichier au format MM_DD_YYYY
        nom_fichier = f"movie_ids_{jour.strftime('%m_%d_%Y')}.json.gz"
        url = f"http://files.tmdb.org/p/exports/{nom_fichier}"
        print(f"🔎 Essai : {url}")

        # 3. Téléchargement
        response = requests.get(url, timeout=60)

        # 4. Si ça marche, on enregistre le fichier
        if response.status_code == 200:
            with open(nom_fichier, "wb") as f:
                f.write(response.content)
            print(f"✅ Téléchargé : {nom_fichier} ({len(response.content) / 1e6:.1f} Mo)")
            return nom_fichier, jour

        print(f"   Pas disponible (code {response.status_code}), on essaie la veille")

    print("❌ Aucun fichier trouvé")
    return None, None


fichier_gz, date_export = telecharger_export_films()

🔎 Essai : http://files.tmdb.org/p/exports/movie_ids_11_05_2026.json.gz
✅ Téléchargé : movie_ids_11_05_2026.json.gz (28.4 Mo)


In [ ]:
import gzip

# Ouvrir le fichier compressé en mode texte et lire toutes les lignes
with gzip.open(fichier_gz, "rt", encoding="utf-8") as f:
    lignes = f.readlines()

print(f"Nombre de lignes : {len(lignes):,}")
print("\nLes 5 premières lignes :")
for ligne in lignes[:5]:
    print(ligne.strip())

Nombre de lignes : 1,269,754

Les 5 premières lignes :
{"adult":false,"id":3924,"original_title":"Blondie","popularity":2.7419,"video":false}
{"adult":false,"id":8773,"original_title":"L'Amour à vingt ans","popularity":1.1479,"video":false}
{"adult":false,"id":25449,"original_title":"New World Disorder 9: Never Enough","popularity":0.7748,"video":false}
{"adult":false,"id":31975,"original_title":"Sesame Street: Elmo Loves You!","popularity":0.2036,"video":true}
{"adult":false,"id":2,"original_title":"Ariel","popularity":3.5943,"video":false}


## 8. Transformation en array JSON et création du DataFrame
Le fichier est au format JSON Lines (un objet JSON par ligne, sans crochets ni virgules).
On convertit chaque ligne en dictionnaire avec json.loads(), on regroupe le tout dans
une liste (array JSON) que l'on enregistre, puis on crée un DataFrame pandas.

In [ ]:
import json

# 1. Chaque ligne de texte devient un dictionnaire Python
films = [json.loads(ligne) for ligne in lignes if ligne.strip()]

print("Type :", type(films))
print("Nombre de films :", f"{len(films):,}")
print("Premier élément :", films[0])
print("Type du premier élément :", type(films[0]))

# 2. On enregistre la liste dans un vrai fichier JSON (format array)
fichier_json = fichier_gz.replace(".json.gz", ".json")
with open(fichier_json, "w", encoding="utf-8") as f:
    json.dump(films, f, ensure_ascii=False)

# 3. On vérifie : le fichier commence maintenant par un crochet
with open(fichier_json, encoding="utf-8") as f:
    print("\nDébut du fichier enregistré :", f.read(200), "...")

Type : <class 'list'>
Nombre de films : 1,269,754
Premier élément : {'adult': False, 'id': 3924, 'original_title': 'Blondie', 'popularity': 2.7419, 'video': False}
Type du premier élément : <class 'dict'>

Début du fichier enregistré : [{"adult": false, "id": 3924, "original_title": "Blondie", "popularity": 2.7419, "video": false}, {"adult": false, "id": 8773, "original_title": "L'Amour à vingt ans", "popularity": 1.1479, "video": f ...


In [ ]:
import pandas as pd

df = pd.DataFrame(films)

print("Dimensions (lignes, colonnes) :", df.shape)
print()
df.info()

Dimensions (lignes, colonnes) : (1269754, 5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1269754 entries, 0 to 1269753
Data columns (total 5 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   adult           1269754 non-null  bool   
 1   id              1269754 non-null  int64  
 2   original_title  1269754 non-null  object 
 3   popularity      1269754 non-null  float64
 4   video           1269748 non-null  object 
dtypes: bool(1), float64(1), int64(1), object(2)
memory usage: 40.0+ MB


In [ ]:
# Aperçu
display(df.head())

# Vérifications de qualité
print("Doublons sur l'id :", df["id"].duplicated().sum())
print("Valeurs manquantes :\n", df.isna().sum())

# Quelques chiffres
print("\nFilms pour adultes :", df["adult"].sum())
print("Vidéos (pas des films de cinéma) :", df["video"].sum())

# Les 10 films les plus populaires du jour
display(df.sort_values("popularity", ascending=False).head(10))

,adult,id,original_title,popularity,video
0,False,3924,Blondie,2.7419,False
1,False,8773,L'Amour à vingt ans,1.1479,False
2,False,25449,New World Disorder 9: Never Enough,0.7748,False
3,False,31975,Sesame Street: Elmo Loves You!,0.2036,True
4,False,2,Ariel,3.5943,False


Doublons sur l'id : 0
Valeurs manquantes :
 adult             0
id                0
original_title    0
popularity        0
video             6
dtype: int64

Films pour adultes : 0
Vidéos (pas des films de cinéma) : 65575


,adult,id,original_title,popularity,video
120,False,155,The Dark Knight,2303.0171,False
558753,False,791523,Wildwood,1088.2974,False
993797,False,1423191,Resident Evil,406.6270,False
739782,False,1058424,호프,395.4968,False
1054868,False,1507696,Apocalipsis Z: Nuevo mundo,373.8924,False
680577,False,969681,Spider-Man: Brand New Day,306.0813,False
228924,False,354912,Coco,261.2047,False
1119211,False,1596275,Drive – The Pretenders,231.4612,False
1266727,False,1813355,A Story of Derry,231.1489,False
1160781,False,1657630,El despertar,210.0108,False


## 9. Injection dans une base de données relationnelle (SQLite)
On utilise SQLite (base stockée dans un fichier, incluse dans Python).
On crée une table `movies` avec `id` comme clé primaire, puis on y insère le DataFrame.
Les booléens sont convertis en 0/1 (SQLite n'a pas de type booléen) et on ajoute
la date de l'export pour savoir de quand datent les données.
Nettoyage : quelques films n'ont pas de valeur pour adult/video ; on les remplace par False
(information absente = non) avant la conversion en 0/1.

In [ ]:
# Copie pour ne pas modifier le DataFrame d'origine
df_sql = df.copy()

# Quelques films n'ont pas de valeur pour adult/video : on considère que c'est "non" (False)
print("Valeurs manquantes avant nettoyage :\n", df_sql[["adult", "video"]].isna().sum())

# SQLite n'a pas de type booléen : True/False → 1/0
df_sql["adult"] = df_sql["adult"].fillna(False).astype(int)
df_sql["video"] = df_sql["video"].fillna(False).astype(int)

# On note la date de l'export (utile pour les mises à jour)
df_sql["export_date"] = date_export.isoformat()

# Sécurité : on retire d'éventuels doublons d'id (la clé primaire les refuserait)
df_sql = df_sql.drop_duplicates(subset="id")

# On met les colonnes dans l'ordre de la future table
df_sql = df_sql[["id", "original_title", "popularity", "adult", "video", "export_date"]]

display(df_sql.head())

Valeurs manquantes avant nettoyage :
 adult    0
video    6
dtype: int64


/tmp/ipykernel_2964/1356866437.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_sql["video"] = df_sql["video"].fillna(False).astype(int)


,id,original_title,popularity,adult,video,export_date
0,3924,Blondie,2.7419,0,0,2026-11-05
1,8773,L'Amour à vingt ans,1.1479,0,0,2026-11-05
2,25449,New World Disorder 9: Never Enough,0.7748,0,0,2026-11-05
3,31975,Sesame Street: Elmo Loves You!,0.2036,0,1,2026-11-05
4,2,Ariel,3.5943,0,0,2026-11-05


In [ ]:
import sqlite3

# Se connecter à la base (le fichier est créé s'il n'existe pas)
conn = sqlite3.connect("tmdb.db")

# Créer la table avec du SQL
conn.execute("""
CREATE TABLE IF NOT EXISTS movies (
    id             INTEGER PRIMARY KEY,
    original_title TEXT,
    popularity     REAL,
    adult          INTEGER,
    video          INTEGER,
    export_date    TEXT
)
""")

# On vide la table au cas où on relance la cellule (pour éviter les doublons)
conn.execute("DELETE FROM movies")
conn.commit()

print("✅ Table movies prête")

✅ Table movies prête


In [ ]:
df_sql.to_sql("movies", conn, if_exists="append", index=False, chunksize=50_000)
conn.commit()

nb = conn.execute("SELECT COUNT(*) FROM movies").fetchone()[0]
print(f"✅ {nb:,} films insérés dans la table movies")

✅ 1,269,754 films insérés dans la table movies


In [ ]:
# Les 10 films les plus populaires
display(pd.read_sql("""
    SELECT id, original_title, popularity
    FROM movies
    ORDER BY popularity DESC
    LIMIT 10
""", conn))

# Retrouver Fight Club grâce à la clé primaire
display(pd.read_sql("SELECT * FROM movies WHERE id = 550", conn))

# Quelques statistiques
display(pd.read_sql("""
    SELECT COUNT(*)                  AS nb_films,
           SUM(adult)                AS nb_adultes,
           SUM(video)                AS nb_videos,
           ROUND(AVG(popularity), 2) AS popularite_moyenne
    FROM movies
""", conn))

,id,original_title,popularity
0,155,The Dark Knight,2303.0171
1,791523,Wildwood,1088.2974
2,1423191,Resident Evil,406.6270
3,1058424,호프,395.4968
4,1507696,Apocalipsis Z: Nuevo mundo,373.8924
5,969681,Spider-Man: Brand New Day,306.0813
6,354912,Coco,261.2047
7,1596275,Drive – The Pretenders,231.4612
8,1813355,A Story of Derry,231.1489
9,1657630,El despertar,210.0108


,id,original_title,popularity,adult,video,export_date
0,550,Fight Club,42.3856,0,0,2026-11-05


,nb_films,nb_adultes,nb_videos,popularite_moyenne
0,1269754,0,65575,0.78


## 10. Mise à jour automatique de la table tous les 3 mois
Les exports TMDB sont supprimés au bout de 3 mois : il faut donc mettre la base à jour
au moins une fois par trimestre. Le script update_tmdb.py :
1. vérifie si le dernier import date de plus de 90 jours (sinon il ne fait rien, sauf --force) ;
2. télécharge l'export le plus récent, le transforme et le nettoie ;
3. le charge dans une table temporaire (staging), puis remplace le contenu de `movies`
   dans une transaction (tout ou rien : la table n'est jamais laissée vide en cas d'erreur) ;
4. enregistre chaque mise à jour dans une table `import_log` (films ajoutés / supprimés).

In [ ]:
%%writefile update_tmdb.py
"""
Mise à jour trimestrielle de la table `movies` à partir des exports quotidiens TMDB.

Usage :
    python update_tmdb.py           -> met à jour si le dernier import date de plus de 90 jours
    python update_tmdb.py --force   -> force la mise à jour
"""
import gzip
import json
import sqlite3
import sys
from datetime import date, datetime, timedelta

import pandas as pd
import requests

DB_PATH = "tmdb.db"
INTERVALLE_JOURS = 90


def telecharger_export_films(nb_jours_max=7):
    """Télécharge l'export le plus récent (aujourd'hui, sinon les jours précédents)."""
    for decalage in range(nb_jours_max + 1):
        jour = date.today() - timedelta(days=decalage)
        nom_fichier = f"movie_ids_{jour.strftime('%m_%d_%Y')}.json.gz"
        url = f"http://files.tmdb.org/p/exports/{nom_fichier}"
        response = requests.get(url, timeout=60)
        if response.status_code == 200:
            with open(nom_fichier, "wb") as f:
                f.write(response.content)
            print(f"✅ Export téléchargé : {nom_fichier}")
            return nom_fichier, jour
        print(f"   Pas d'export pour le {jour} (code {response.status_code})")
    raise RuntimeError("Aucun export trouvé")


def preparer_dataframe(fichier_gz, date_export):
    """JSON Lines -> array JSON -> DataFrame nettoyé."""
    with gzip.open(fichier_gz, "rt", encoding="utf-8") as f:
        films = [json.loads(ligne) for ligne in f if ligne.strip()]
    df = pd.DataFrame(films)
    df["adult"] = df["adult"].fillna(False).astype(int)
    df["video"] = df["video"].fillna(False).astype(int)
    df["export_date"] = date_export.isoformat()
    df = df.drop_duplicates(subset="id")
    return df[["id", "original_title", "popularity", "adult", "video", "export_date"]]


def creer_tables(conn):
    conn.execute("""
        CREATE TABLE IF NOT EXISTS movies (
            id             INTEGER PRIMARY KEY,
            original_title TEXT,
            popularity     REAL,
            adult          INTEGER,
            video          INTEGER,
            export_date    TEXT
        )""")
    conn.execute("""
        CREATE TABLE IF NOT EXISTS import_log (
            date_maj     TEXT,
            date_export  TEXT,
            nb_films     INTEGER,
            nb_ajoutes   INTEGER,
            nb_supprimes INTEGER
        )""")
    conn.commit()


def derniere_maj(conn):
    resultat = conn.execute("SELECT MAX(date_maj) FROM import_log").fetchone()[0]
    return datetime.fromisoformat(resultat) if resultat else None


def mettre_a_jour(conn, df, date_export):
    """Staging + transaction : la table movies n'est jamais laissée vide."""
    # 1. Charger les nouvelles données dans une table temporaire
    df.to_sql("movies_staging", conn, if_exists="replace", index=False, chunksize=50_000)

    # 2. Tout le reste en une seule transaction (tout ou rien)
    with conn:
        nb_ajoutes = conn.execute(
            "SELECT COUNT(*) FROM movies_staging WHERE id NOT IN (SELECT id FROM movies)"
        ).fetchone()[0]
        nb_supprimes = conn.execute(
            "SELECT COUNT(*) FROM movies WHERE id NOT IN (SELECT id FROM movies_staging)"
        ).fetchone()[0]

        conn.execute("DELETE FROM movies")
        conn.execute("INSERT INTO movies SELECT * FROM movies_staging")
        conn.execute("DROP TABLE movies_staging")
        conn.execute(
            "INSERT INTO import_log VALUES (?, ?, ?, ?, ?)",
            (datetime.now().isoformat(timespec="seconds"), date_export.isoformat(),
             len(df), nb_ajoutes, nb_supprimes),
        )
    return nb_ajoutes, nb_supprimes


def main():
    forcer = "--force" in sys.argv
    conn = sqlite3.connect(DB_PATH)
    try:
        creer_tables(conn)

        derniere = derniere_maj(conn)
        if derniere and not forcer and datetime.now() - derniere < timedelta(days=INTERVALLE_JOURS):
            print(f"ℹ️ Dernière mise à jour le {derniere.date()} (moins de {INTERVALLE_JOURS} jours) : rien à faire.")
            return

        fichier_gz, date_export = telecharger_export_films()
        df = preparer_dataframe(fichier_gz, date_export)
        nb_ajoutes, nb_supprimes = mettre_a_jour(conn, df, date_export)
        print(f"✅ Table movies à jour : {len(df):,} films (+{nb_ajoutes:,} / -{nb_supprimes:,})")
    except Exception as erreur:
        print(f"❌ Échec de la mise à jour : {erreur}")
        sys.exit(1)
    finally:
        conn.close()


if __name__ == "__main__":
    main()

Writing update_tmdb.py


In [ ]:
!python update_tmdb.py --force

✅ Export téléchargé : movie_ids_11_05_2026.json.gz
/content/update_tmdb.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["video"] = df["video"].fillna(False).astype(int)
✅ Table movies à jour : 1,269,754 films (+0 / -0)


In [ ]:
!python update_tmdb.py

ℹ️ Dernière mise à jour le 2026-11-05 (moins de 90 jours) : rien à faire.


In [ ]:
display(pd.read_sql("SELECT * FROM import_log ORDER BY date_maj DESC", conn))

,date_maj,date_export,nb_films,nb_ajoutes,nb_supprimes
0,2026-11-05T11:11:47,2026-11-05,1269754,0,0


### Planification automatique (tous les 3 mois)
Colab ne permet pas de lancer un script automatiquement : en production, on place
update_tmdb.py sur un serveur ou un ordinateur et on le programme :

- Linux / macOS, avec cron (commande `crontab -e`) :
  0 3 1 */3 *  cd /chemin/vers/projet && python3 update_tmdb.py >> update.log 2>&1
  → le 1er janvier, avril, juillet et octobre à 3h du matin.

- Windows, avec le Planificateur de tâches :
  Créer une tâche → Déclencheur « Mensuel » → mois : janvier, avril, juillet, octobre, jour 1, 03:00
  → Action : python.exe, argument : update_tmdb.py, dossier : celui du projet.

Le script vérifie lui-même que 90 jours se sont écoulés : il peut donc être planifié
plus souvent (par exemple chaque mois) sans risque de mises à jour inutiles.

## 11. Interface Streamlit
Application web qui présente les retours de l'API TMDB :
- recherche d'un film par titre et affichage de sa fiche (affiche, note, genres, résumé) ;
- fiche d'un film à partir de son identifiant (équivalent visuel de get_movie_details) ;
- exploration de la base SQL (top des films populaires, historique des mises à jour).
Les appels réutilisent la gestion d'erreurs (Q1) et un cache d'1 heure (Q3, via st.cache_data).

In [ ]:
!pip install -q streamlit

In [ ]:
%%writefile app.py
import os
import sqlite3
import time

import pandas as pd
import requests
import streamlit as st

# ---------- Configuration ----------
API_KEY = os.getenv("TMDB_API_KEY")          # la clé est passée par une variable d'environnement
BASE_URL = "https://api.themoviedb.org/3"
IMG_URL = "https://image.tmdb.org/t/p/w500"
DB_PATH = "tmdb.db"
CODES_TEMPORAIRES = {429, 500, 502, 503, 504}
MESSAGES_ERREURS = {
    400: "Requête mal formée (400)",
    401: "Clé API absente ou invalide (401)",
    403: "Accès interdit (403)",
    404: "Film introuvable (404)",
}

st.set_page_config(page_title="Explorateur TMDB", page_icon="🎬", layout="wide")


# ---------- Appels API : robustes (Q1) + cache d'1 heure (Q3) ----------
@st.cache_data(ttl=3600, show_spinner=False)
def requete_robuste(url, params, max_tentatives=5):
    """Retry + backoff sur les erreurs temporaires, exception sur les erreurs définitives.
    Les exceptions ne sont pas mises en cache par Streamlit : seuls les succès le sont."""
    for tentative in range(1, max_tentatives + 1):
        attente = 2 ** (tentative - 1)
        try:
            response = requests.get(url, params=params, timeout=10)
        except requests.exceptions.RequestException:
            time.sleep(attente)
            continue
        if response.status_code == 200:
            return response.json()
        if response.status_code in CODES_TEMPORAIRES:
            time.sleep(attente)
            continue
        raise RuntimeError(MESSAGES_ERREURS.get(response.status_code, f"Erreur {response.status_code}"))
    raise RuntimeError("Échec après plusieurs tentatives")


def afficher_fiche(movie_id):
    """Affiche la fiche complète d'un film."""
    try:
        data = requete_robuste(f"{BASE_URL}/movie/{movie_id}",
                               {"api_key": API_KEY, "language": "fr-FR"})
    except RuntimeError as erreur:
        st.error(f"❌ {erreur}")
        return

    col_affiche, col_infos = st.columns([1, 2])
    with col_affiche:
        if data.get("poster_path"):
            st.image(IMG_URL + data["poster_path"])
        else:
            st.info("Pas d'affiche disponible")
    with col_infos:
        st.header(data.get("title", "Titre inconnu"))
        if data.get("original_title") != data.get("title"):
            st.caption(f"Titre original : {data.get('original_title')}")
        st.write(f"📅 **Date de sortie :** {data.get('release_date') or 'inconnue'}")
        genres = ", ".join(g["name"] for g in data.get("genres", []))
        st.write(f"🎭 **Genres :** {genres or 'non renseignés'}")

        m1, m2, m3 = st.columns(3)
        m1.metric("⭐ Note moyenne", f"{data.get('vote_average', 0):.1f} / 10")
        m2.metric("🗳️ Votes", f"{data.get('vote_count', 0):,}")
        m3.metric("🔥 Popularité", f"{data.get('popularity', 0):.1f}")

        st.subheader("Résumé")
        st.write(data.get("overview") or "Pas de résumé disponible.")


# ---------- Interface ----------
st.title("🎬 Explorateur TMDB")
st.caption("Données issues de l'API The Movie Database : appels robustes (retry) et mis en cache 1 h.")

if not API_KEY:
    st.error("Clé API introuvable : définis la variable d'environnement TMDB_API_KEY.")
    st.stop()

onglet_recherche, onglet_id, onglet_bd = st.tabs(
    ["🔎 Rechercher un film", "🎞️ Fiche par identifiant", "🗄️ Base de données"]
)

# Onglet 1 : recherche par titre
with onglet_recherche:
    titre = st.text_input("Titre du film", placeholder="ex. Fight Club, Inception, Amélie…")
    if titre:
        try:
            resultats = requete_robuste(f"{BASE_URL}/search/movie",
                                        {"api_key": API_KEY, "language": "fr-FR", "query": titre})
        except RuntimeError as erreur:
            st.error(f"❌ {erreur}")
            resultats = {"results": []}

        films = resultats.get("results", [])[:10]
        if not films:
            st.warning("Aucun film trouvé.")
        else:
            options = {f"{f['title']} ({(f.get('release_date') or '????')[:4]})": f["id"] for f in films}
            choix = st.selectbox(f"{len(films)} résultat(s) — choisis un film :", list(options))
            st.divider()
            afficher_fiche(options[choix])

# Onglet 2 : fiche par identifiant (get_movie_details)
with onglet_id:
    movie_id = st.number_input("Identifiant TMDB du film", min_value=1, value=550, step=1)
    afficher_fiche(int(movie_id))

# Onglet 3 : la base SQL alimentée par l'export quotidien
with onglet_bd:
    if not os.path.exists(DB_PATH):
        st.warning("Base tmdb.db introuvable : exécute d'abord les étapes 7 à 10 du notebook.")
    else:
        conn = sqlite3.connect(DB_PATH)
        stats = pd.read_sql(
            "SELECT COUNT(*) AS nb, SUM(adult) AS adultes, SUM(video) AS videos, "
            "MAX(export_date) AS date_export FROM movies", conn
        ).iloc[0]

        c1, c2, c3, c4 = st.columns(4)
        c1.metric("Films en base", f"{int(stats['nb']):,}")
        c2.metric("Films adultes", f"{int(stats['adultes'] or 0):,}")
        c3.metric("Vidéos", f"{int(stats['videos'] or 0):,}")
        c4.metric("Date de l'export", stats["date_export"] or "-")

        n = st.slider("Nombre de films à afficher", 5, 50, 15)
        top = pd.read_sql(
            "SELECT id, original_title, popularity FROM movies "
            "WHERE adult = 0 ORDER BY popularity DESC LIMIT ?", conn, params=(n,)
        )
        st.subheader(f"Top {n} des films les plus populaires")
        st.bar_chart(top, x="original_title", y="popularity", horizontal=True)
        st.dataframe(top, hide_index=True, use_container_width=True)

        st.subheader("Historique des mises à jour")
        try:
            st.dataframe(pd.read_sql("SELECT * FROM import_log ORDER BY date_maj DESC", conn),
                         hide_index=True, use_container_width=True)
        except Exception:
            st.info("Aucune mise à jour enregistrée pour l'instant.")
        conn.close()

Writing app.py


In [ ]:
import time

# 1. Télécharger l'outil de tunnel de Cloudflare (une seule fois)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# 2. Ouvrir le tunnel vers Streamlit (port 8501), en arrière-plan
!nohup ./cloudflared tunnel --url http://localhost:8501 > cloudflared.log 2>&1 &

# 3. Attendre que le tunnel démarre, puis afficher l'adresse
time.sleep(8)
print("👉 Ouvre ce lien :")
!grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' cloudflared.log | head -1

cloudflared: Text file busy
👉 Ouvre ce lien :
https://achieve-trying-summaries-gaming.trycloudflare.com


In [ ]:
import os, time
from google.colab import userdata

# 1. Remettre la clé dans les variables d'environnement (par sécurité)
os.environ["TMDB_API_KEY"] = userdata.get("TMDB_API_KEY")

# 2. Arrêter un éventuel ancien Streamlit, puis le relancer en arrière-plan
!pkill -f "streamlit run"
!nohup streamlit run app.py --server.port 8501 --server.headless true > /content/logs.txt 2>&1 &

# 3. Attendre qu'il démarre, puis vérifier qu'il répond
time.sleep(6)
print("Réponse de Streamlit (200 = OK) :")
!curl -s -o /dev/null -w "%{http_code}" http://localhost:8501

Réponse de Streamlit (200 = OK) :
200